In [3]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# download the weights 
weights = ResNet18_Weights.DEFAULT
transform = weights.transforms()

In [6]:
# load the datasets
train_dataset = datasets.CIFAR10(
    root="../data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="../data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [7]:
model = resnet18(weights=weights)

model.fc = nn.Linear(
    model.fc.in_features,
    len(train_dataset.classes)
)

model = model.to(device)

In [8]:
# Freeze every parameter first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the last residual stage
for param in model.layer4.parameters():
    param.requires_grad = True

# Unfreeze the new classifier
for param in model.fc.parameters():
    param.requires_grad = True

In [10]:
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 11,181,642
Trainable parameters: 8,398,858


In [13]:
loss_fn = nn.CrossEntropyLoss()

#different learning rates for different layers
optimizer = torch.optim.Adam([
    {
        "params": model.layer4.parameters(),
        "lr": 0.0001
    },
    {
        "params": model.fc.parameters(),
        "lr": 0.001
    }
])

In [14]:
train_losses = []

for epoch in range(5):
    model.train()

    # Keep all BatchNorm running statistics fixed
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    epoch_losses = []

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        output = model(images)
        loss = loss_fn(output, labels)

        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())

    average_loss = sum(epoch_losses) / len(epoch_losses)
    train_losses.append(average_loss)

    print(
        f"Epoch {epoch + 1} -- "
        f"Average Loss: {average_loss:.4f}"
    )

Epoch 1 -- Average Loss: 0.3951
Epoch 2 -- Average Loss: 0.1904
Epoch 3 -- Average Loss: 0.0991
Epoch 4 -- Average Loss: 0.0611
Epoch 5 -- Average Loss: 0.0447


In [18]:
def cache_layer3_outputs(loader, dataset_size, model, device):
    cached_features = torch.empty(
        (dataset_size, 256, 14, 14),
        dtype=torch.float32
    )

    cached_labels = torch.empty(
        dataset_size,
        dtype=torch.long
    )

    model.eval()
    offset = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            x = model.conv1(images)
            x = model.bn1(x)
            x = model.relu(x)
            x = model.maxpool(x)

            x = model.layer1(x)
            x = model.layer2(x)
            x = model.layer3(x)

            batch_size = images.size(0)

            cached_features[offset:offset + batch_size].copy_(
                x.cpu()
            )
            cached_labels[offset:offset + batch_size].copy_(
                labels
            )

            offset += batch_size

    if offset != dataset_size:
        raise RuntimeError(
            f"Expected {dataset_size} samples, got {offset}"
        )

    return cached_features, cached_labels

In [ ]:
train_layer3_features, train_cached_labels = cache_layer3_outputs(
    train_loader,
    len(train_dataset),
    model,
    device
)

test_layer3_features, test_cached_labels = cache_layer3_outputs(
    test_loader,
    len(test_dataset),
    model,
    device
)